# SECOM 공식 TabDDPM Colab 실험

Yandex Research의 공식 `tab-ddpm` 저장소를 이용해 SECOM 센서 데이터를 label-conditioned TabDDPM으로 학습합니다. test 20%는 모델 선택에 사용하지 않습니다. UCI 원본은 자동으로 내려받으므로 Kaggle 토큰이나 수동 업로드가 필요 없습니다.

실행 전 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. 먼저 `RUN_MODE='smoke'`로 전체 작동을 확인한 뒤, `RUN_MODE='paper'`로 바꾸고 **런타임 → 세션 다시 시작 후 모두 실행**하여 정식 결과를 만드세요. 마지막의 `secom_tabddpm_results.zip`을 프로젝트로 가져오면 됩니다.

In [ ]:
%pip install -q catboost xgboost
import json, os, shutil, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from google.colab import drive, files
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다. 런타임 유형을 GPU로 변경하세요.'
print('GPU:', torch.cuda.get_device_name(0))
print('Python:', sys.version)
print('Torch:', torch.__version__)

## 1. 실험 설정
`paper`는 정식 비교용 30,000 step입니다. 먼저 `smoke`로 전체 셀의 작동을 확인하고, 제출 결과는 `paper`로 새 세션에서 다시 실행하세요. `SAVE_TO_DRIVE=True`이면 결과 ZIP이 Google Drive에도 복사됩니다.

In [ ]:
RUN_MODE = 'paper'  # 'smoke' 또는 'paper'
SAVE_TO_DRIVE = True
RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE_WITHIN_TRAIN = 0.20
TRAIN_STEPS = 2_000 if RUN_MODE == 'smoke' else 30_000
DIFFUSION_TIMESTEPS = 100
SYNTH_RATIOS = (0.25, 0.50, 0.75, 1.00)
ROOT = Path('/content/secom_tabddpm')
INPUT_DIR = ROOT / 'input'
OUTPUT_DIR = ROOT / 'output' / RUN_MODE
for directory in (INPUT_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR = None
if SAVE_TO_DRIVE:
    drive.mount('/content/drive')
    DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/SECOM_TabDDPM')
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('mode=', RUN_MODE, 'train_steps=', TRAIN_STEPS)

## 2. UCI SECOM 원본 자동 다운로드
UCI 공식 ZIP에서 `secom.data`와 `secom_labels.data`를 자동으로 받습니다. 네트워크 오류가 날 때만 파일 선택 창이 열립니다.

In [ ]:
required = {'secom.data', 'secom_labels.data'}
UCI_ZIP_URL = 'https://archive.ics.uci.edu/static/public/179/secom.zip'
try:
    archive_path = ROOT / 'secom.zip'
    urllib.request.urlretrieve(UCI_ZIP_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        members = {Path(member).name: member for member in archive.namelist()}
        missing = required - set(members)
        assert not missing, f'UCI ZIP에 빠진 파일: {sorted(missing)}'
        for name in required:
            (INPUT_DIR / name).write_bytes(archive.read(members[name]))
    print('UCI 자동 다운로드 완료:', sorted(required))
except Exception as error:
    print('자동 다운로드 실패:', error)
    print('두 원본 파일을 직접 선택하세요.')
    uploaded = files.upload()
    missing = required - set(uploaded)
    assert not missing, f'빠진 파일: {sorted(missing)}'
    for name in required:
        (INPUT_DIR / name).write_bytes(uploaded[name])
assert all((INPUT_DIR / name).stat().st_size > 0 for name in required)
print('입력 준비 완료:', INPUT_DIR)

## 3. 공식 TabDDPM 격리 환경 준비
공식 저장소가 지정한 구버전 의존성을 현재 Colab과 섞지 않도록 별도의 Miniconda 환경을 만듭니다. 최초 설치에는 시간이 걸립니다.

In [ ]:
OFFICIAL_REPO = ROOT / 'tab-ddpm'
CONDA_DIR = ROOT / 'miniconda'
ENV_PYTHON = CONDA_DIR / 'envs' / 'tabddpm' / 'bin' / 'python'
if not OFFICIAL_REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/yandex-research/tab-ddpm.git', str(OFFICIAL_REPO)], check=True)
official_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=OFFICIAL_REPO, text=True).strip()
if not (CONDA_DIR / 'bin' / 'conda').exists():
    installer = ROOT / 'miniconda.sh'
    subprocess.run(['wget', '-q', 'https://repo.anaconda.com/miniconda/Miniconda3-py39_24.3.0-0-Linux-x86_64.sh', '-O', str(installer)], check=True)
    subprocess.run(['bash', str(installer), '-b', '-p', str(CONDA_DIR)], check=True)
if not ENV_PYTHON.exists():
    subprocess.run([str(CONDA_DIR / 'bin' / 'conda'), 'create', '-y', '-n', 'tabddpm', 'python=3.9.7', 'pip'], check=True)
marker = ROOT / '.official_dependencies_ready'
if not marker.exists():
    subprocess.run([str(ENV_PYTHON), '-m', 'pip', 'install', 'torch==1.10.1+cu111', '--extra-index-url', 'https://download.pytorch.org/whl/cu111'], check=True)
    subprocess.run([str(ENV_PYTHON), '-m', 'pip', 'install', '-r', str(OFFICIAL_REPO / 'requirements.txt')], check=True)
    marker.write_text('ok')
env = os.environ.copy()
env['PYTHONPATH'] = str(OFFICIAL_REPO)
env['PROJECT_DIR'] = str(OFFICIAL_REPO)
gpu_check = subprocess.run([str(ENV_PYTHON), '-c', "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"], env=env, text=True, capture_output=True, check=True)
print('공식 저장소 commit:', official_commit)
print('공식 환경 GPU 확인:', gpu_check.stdout.strip())

## 4. 동일한 SECOM 전처리와 고정 분할 생성
라벨 -1/1을 0/1로 변환하고 결측률 50% 이상 변수와 상수를 제거합니다. median은 train에만 맞춥니다.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
X_raw = pd.read_csv(INPUT_DIR / 'secom.data', sep=r'\s+', header=None)
labels = pd.read_csv(INPUT_DIR / 'secom_labels.data', sep=r'\s+', header=None)
assert X_raw.shape == (1567, 590), X_raw.shape
raw_y = labels.iloc[:, 0].astype(int)
assert set(raw_y.unique()) == {-1, 1}
y = raw_y.map({-1: 0, 1: 1}).astype('int64')
X_raw.columns = [f'feature_{i}' for i in range(590)]
high_missing = X_raw.columns[X_raw.isna().mean() >= 0.5].tolist()
X = X_raw.drop(columns=high_missing)
constants = X.columns[X.nunique(dropna=True) <= 1].tolist()
X = X.drop(columns=constants)
assert X.shape[1] == 446, X.shape
indices = np.arange(len(y))
trainval_idx, test_idx = train_test_split(indices, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
train_idx, val_idx = train_test_split(trainval_idx, test_size=VALIDATION_SIZE_WITHIN_TRAIN, random_state=RANDOM_STATE, stratify=y.iloc[trainval_idx])
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X.iloc[train_idx]).astype('float32')
X_val = imputer.transform(X.iloc[val_idx]).astype('float32')
X_test = imputer.transform(X.iloc[test_idx]).astype('float32')
y_train = y.iloc[train_idx].to_numpy()
y_val = y.iloc[val_idx].to_numpy()
y_test = y.iloc[test_idx].to_numpy()
DATASET_DIR = OFFICIAL_REPO / 'data' / 'secom'
DATASET_DIR.mkdir(parents=True, exist_ok=True)
for split, features, target in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    np.save(DATASET_DIR / f'X_num_{split}.npy', features)
    np.save(DATASET_DIR / f'y_{split}.npy', target)
info = {'name': 'secom', 'id': 'secom', 'task_type': 'binclass', 'n_num_features': 446, 'n_cat_features': 0, 'train_size': len(y_train), 'val_size': len(y_val), 'test_size': len(y_test), 'n_classes': 2}
(DATASET_DIR / 'info.json').write_text(json.dumps(info, indent=2))
(OUTPUT_DIR / 'selected_features.json').write_text(json.dumps(X.columns.tolist(), indent=2))
print('train/val/test:', len(y_train), len(y_val), len(y_test))
print('defects:', int(y_train.sum()), int(y_val.sum()), int(y_test.sum()))

## 5. TabDDPM 설정 작성 및 학습·생성
수치형 변수는 quantile normalization, label은 조건부 생성입니다. `disbalance='fill'`은 부족한 불량 클래스만 생성합니다.

In [ ]:
EXP_DIR = OFFICIAL_REPO / 'exp' / 'secom' / f'tabddpm_{RUN_MODE}'
EXP_DIR.mkdir(parents=True, exist_ok=True)
config_text = f'''seed = {RANDOM_STATE}
parent_dir = "{EXP_DIR}"
real_data_path = "{DATASET_DIR}"
model_type = "mlp"
num_numerical_features = 446
device = "cuda:0"

[model_params]
num_classes = 2
is_y_cond = true

[model_params.rtdl_params]
d_layers = [512, 1024, 1024, 1024, 512]
dropout = 0.0

[diffusion_params]
num_timesteps = {DIFFUSION_TIMESTEPS}
gaussian_loss_type = "mse"
scheduler = "cosine"

[train.main]
steps = {TRAIN_STEPS}
lr = 0.0008
weight_decay = 0.0
batch_size = 4096

[train.T]
seed = {RANDOM_STATE}
normalization = "quantile"
num_nan_policy = "__none__"
cat_nan_policy = "__none__"
cat_min_frequency = "__none__"
cat_encoding = "__none__"
y_policy = "default"

[sample]
num_samples = {len(y_train)}
batch_size = 2048
seed = {RANDOM_STATE}
disbalance = "fill"

[eval.type]
eval_model = "catboost"
eval_type = "synthetic"

[eval.T]
seed = {RANDOM_STATE}
normalization = "__none__"
num_nan_policy = "__none__"
cat_nan_policy = "__none__"
cat_min_frequency = "__none__"
cat_encoding = "__none__"
y_policy = "default"
'''
CONFIG_PATH = EXP_DIR / 'config.toml'
CONFIG_PATH.write_text(config_text)
started = time.perf_counter()
subprocess.run([str(ENV_PYTHON), 'scripts/pipeline.py', '--config', str(CONFIG_PATH), '--train', '--sample'], cwd=OFFICIAL_REPO, env=env, check=True)
training_seconds = time.perf_counter() - started
print(f'TabDDPM 학습·생성 완료: {training_seconds/60:.1f}분')

## 6. 합성 품질 및 실제 validation/test 평가
전략 선택은 validation PR-AUC로만 수행합니다. test는 선택 후 참고용으로 한 번 평가합니다.

In [ ]:
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, precision_score, recall_score, f1_score, fbeta_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
X_synth_all = np.load(EXP_DIR / 'X_num_train.npy').astype('float32')
y_synth_all = np.load(EXP_DIR / 'y_train.npy').astype(int)
X_synth = X_synth_all[y_synth_all == 1]
assert len(X_synth) > 0, '불량 합성행이 생성되지 않았습니다.'
lower, upper = X_train[y_train == 1].min(0), X_train[y_train == 1].max(0)
X_synth = np.clip(X_synth, lower, upper)
def threshold_f2(y_true, probability):
    p, r, t = precision_recall_curve(y_true, probability)
    score = 5*p[:-1]*r[:-1]/(4*p[:-1]+r[:-1]+1e-12)
    return float(t[int(np.nanargmax(score))]) if len(t) else 0.5
def scores(y_true, probability, threshold):
    prediction = (probability >= threshold).astype(int)
    return {'pr_auc': average_precision_score(y_true, probability), 'roc_auc': roc_auc_score(y_true, probability), 'precision': precision_score(y_true, prediction, zero_division=0), 'recall': recall_score(y_true, prediction, zero_division=0), 'f1': f1_score(y_true, prediction, zero_division=0), 'f2': fbeta_score(y_true, prediction, beta=2, zero_division=0), 'threshold': threshold}
def make_model(name, weighted, target):
    ratio = float((target == 0).sum() / max((target == 1).sum(), 1))
    if name == 'CatBoost':
        return CatBoostClassifier(iterations=400, learning_rate=0.03, depth=6, verbose=False, allow_writing_files=False, random_seed=RANDOM_STATE, scale_pos_weight=ratio if weighted else 1.0)
    return XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=4, min_child_weight=2, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', tree_method='hist', random_state=RANDOM_STATE, scale_pos_weight=ratio if weighted else 1.0)
def augmented(real_X, real_y, ratio):
    needed = max(0, int(round((real_y == 0).sum() * ratio)) - int((real_y == 1).sum()))
    rng = np.random.default_rng(RANDOM_STATE + int(ratio * 100))
    chosen = X_synth[rng.choice(len(X_synth), needed, replace=needed > len(X_synth))]
    return np.vstack([real_X, chosen]), np.concatenate([real_y, np.ones(needed, dtype=int)])
validation_rows = []
strategies = [('scale_pos_weight', None)] + [(f'TabDDPM_{ratio:.2f}', ratio) for ratio in SYNTH_RATIOS]
for model_name in ('CatBoost', 'XGBoost'):
    for strategy, ratio in strategies:
        fit_X, fit_y = (X_train, y_train) if ratio is None else augmented(X_train, y_train, ratio)
        model = make_model(model_name, ratio is None, fit_y)
        model.fit(fit_X, fit_y)
        probability = model.predict_proba(X_val)[:, 1]
        threshold = threshold_f2(y_val, probability)
        validation_rows.append({'model': model_name, 'strategy': strategy, 'train_rows': len(fit_y), **scores(y_val, probability, threshold)})
validation_results = pd.DataFrame(validation_rows).sort_values(['pr_auc', 'f2'], ascending=False)
validation_results.to_csv(OUTPUT_DIR / 'validation_results.csv', index=False)
display(validation_results)
selected = validation_results.iloc[0]
test_rows = []
test_candidates = [('CatBoost', 'scale_pos_weight'), ('XGBoost', 'scale_pos_weight'), (str(selected.model), str(selected.strategy))]
for model_name, strategy in test_candidates:
    ratio = None if strategy == 'scale_pos_weight' else float(strategy.split('_')[1])
    full_X = np.vstack([X_train, X_val])
    full_y = np.concatenate([y_train, y_val])
    fit_X, fit_y = (full_X, full_y) if ratio is None else augmented(full_X, full_y, ratio)
    model = make_model(model_name, ratio is None, fit_y)
    model.fit(fit_X, fit_y)
    probability = model.predict_proba(X_test)[:, 1]
    threshold = float(validation_results.query('model == @model_name and strategy == @strategy').iloc[0].threshold)
    test_rows.append({'model': model_name, 'strategy': strategy, **scores(y_test, probability, threshold)})
test_results = pd.DataFrame(test_rows).drop_duplicates(['model', 'strategy'])
test_results.to_csv(OUTPUT_DIR / 'test_results.csv', index=False)
display(test_results)
real_defect = X_train[y_train == 1]
ks_similarity = float(np.mean([1 - ks_2samp(real_defect[:, i], X_synth[:, i]).statistic for i in range(X_train.shape[1])]))
real_corr = np.nan_to_num(np.corrcoef(real_defect, rowvar=False), nan=0.0)
synth_corr = np.nan_to_num(np.corrcoef(X_synth, rowvar=False), nan=0.0)
corr_similarity = float(max(0.0, 1 - np.mean(np.abs(real_corr - synth_corr))/2))
privacy_scaler = StandardScaler().fit(real_defect)
scaled_real_defect = privacy_scaler.transform(real_defect)
scaled_synth = privacy_scaler.transform(X_synth)
distances = NearestNeighbors(n_neighbors=1).fit(scaled_real_defect).kneighbors(scaled_synth)[0][:, 0]
rng = np.random.default_rng(RANDOM_STATE)
n = min(len(real_defect), len(X_synth))
disc_X = np.vstack([real_defect, X_synth[rng.choice(len(X_synth), n, replace=False)]])
disc_y = np.concatenate([np.zeros(len(real_defect), dtype=int), np.ones(n, dtype=int)])
disc = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
disc_auc = float(cross_val_score(disc, disc_X, disc_y, scoring='roc_auc', cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)).mean())
quality = pd.DataFrame([{'generator': 'official_TabDDPM', 'real_defect_rows': len(real_defect), 'synthetic_defect_rows': len(X_synth), 'ks_similarity_mean': ks_similarity, 'correlation_similarity': corr_similarity, 'discriminator_auc': max(disc_auc, 1-disc_auc), 'exact_copy_rate': float(np.mean(distances < 1e-8)), 'nearest_real_distance_median': float(np.median(distances))}])
quality.to_csv(OUTPUT_DIR / 'synthetic_quality.csv', index=False)
pd.DataFrame(X_synth, columns=X.columns).to_csv(OUTPUT_DIR / 'tabddpm_synthetic_defects.csv', index=False)
display(quality)

## 7. 전달용 ZIP 생성
아래 셀은 결과 ZIP을 다운로드하고, `SAVE_TO_DRIVE=True`이면 Drive의 `MyDrive/AI_project/SECOM_TabDDPM/`에도 보관합니다. 다운로드한 ZIP을 프로젝트의 `결과물/secom/colab_가져오기/`에 넣으세요.

In [ ]:
shutil.copy2(CONFIG_PATH, OUTPUT_DIR / 'config.toml')
shutil.copy2(EXP_DIR / 'model.pt', OUTPUT_DIR / 'tabddpm_model.pt')
metadata = {'run_mode': RUN_MODE, 'random_state': RANDOM_STATE, 'train_steps': TRAIN_STEPS, 'diffusion_timesteps': DIFFUSION_TIMESTEPS, 'raw_rows': len(y), 'retained_features': X.shape[1], 'train_rows': len(y_train), 'validation_rows': len(y_val), 'test_rows': len(y_test), 'training_seconds': training_seconds, 'gpu': torch.cuda.get_device_name(0), 'selection_uses_test': False, 'threshold_metric': 'F2_on_validation', 'validation_protocol': 'fixed_stratified_train_validation_test', 'selected_model': str(selected.model), 'selected_strategy': str(selected.strategy), 'official_repository': 'https://github.com/yandex-research/tab-ddpm', 'official_commit': official_commit, 'dataset_source': UCI_ZIP_URL}
(OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
summary = f'''# SECOM 공식 TabDDPM Colab 결과

- 실행 모드: {RUN_MODE}
- GPU: {metadata['gpu']}
- 학습 step: {TRAIN_STEPS}
- 학습 시간: {training_seconds/60:.1f}분
- validation 선택: {selected.model} / {selected.strategy}
- validation PR-AUC: {selected.pr_auc:.4f}
- validation F2: {selected.f2:.4f}
- 임계값은 validation F2로 선택
- test는 모델·전략·임계값 선택에 사용하지 않음
'''
(OUTPUT_DIR / 'summary.md').write_text(summary, encoding='utf-8')
archive = shutil.make_archive('/content/secom_tabddpm_results', 'zip', OUTPUT_DIR)
if DRIVE_OUTPUT_DIR is not None:
    drive_archive = DRIVE_OUTPUT_DIR / f'secom_tabddpm_results_{RUN_MODE}.zip'
    shutil.copy2(archive, drive_archive)
    print('Drive 백업:', drive_archive)
print('생성 완료:', archive)
files.download(archive)